# 01 - TF-IDF Vectorization
This notebook corresponds to `01-sbert_retraining.ipynb`. Instead of training an encoder, we fit a **ManualTfidfVectorizer** (manually implemented TF-IDF) on the training set using the exact same data split as SBERT, and save the vectorizer for downstream classification.

In [ ]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
import joblib
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../../"))
sys.path.insert(0, PROJECT_ROOT)
from tfidf_helper import ManualTfidfVectorizer


## 1. Load Data and Exact Split from SBERT

In [ ]:
# Load the same split used in SBERT
split_path = os.path.join(PROJECT_ROOT, "outputs", "SBERT", "data_split.npz")
split_data = np.load(split_path)
train_idx, val_idx, test_idx = split_data['train_idx'], split_data['val_idx'], split_data['test_idx']

# Load dataset
df = pd.read_csv(os.path.join(PROJECT_ROOT, "data", "Restaurant_ABSA_processed.csv"))
aspect_cols = ['food', 'service', 'price', 'ambiance', 'miscellaneous']


## 2. Extract Subsets

In [ ]:
# Assuming review_cleaned already exists
X_train = df.iloc[train_idx]['review_cleaned'].fillna("")
y_train = df.iloc[train_idx][aspect_cols]

X_val = df.iloc[val_idx]['review_cleaned'].fillna("")
y_val = df.iloc[val_idx][aspect_cols]

X_test = df.iloc[test_idx]['review_cleaned'].fillna("")
y_test = df.iloc[test_idx][aspect_cols]

print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")

## 3. Fit ManualTfidfVectorizer

In [ ]:
# Manual TF-IDF implementation (replacing sklearn TfidfVectorizer)
vectorizer = ManualTfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True)
vectorizer.fit(X_train)

X_train_tfidf = vectorizer.transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)


## 4. Save Vectorizer and Features

In [ ]:
# Create directories
os.makedirs(os.path.join(PROJECT_ROOT, "models", "TF-IDF"), exist_ok=True)
os.makedirs(os.path.join(PROJECT_ROOT, "outputs", "TF-IDF"), exist_ok=True)

# Save Vectorizer
joblib.dump(vectorizer, os.path.join(PROJECT_ROOT, "models", "TF-IDF", "vectorizer.joblib"))

# Save TF-IDF Sparse Matrices and labels for Notebook 2
sp.save_npz(os.path.join(PROJECT_ROOT, "outputs", "TF-IDF", "X_train_tfidf.npz"), X_train_tfidf)
sp.save_npz(os.path.join(PROJECT_ROOT, "outputs", "TF-IDF", "X_val_tfidf.npz"), X_val_tfidf)
sp.save_npz(os.path.join(PROJECT_ROOT, "outputs", "TF-IDF", "X_test_tfidf.npz"), X_test_tfidf)

y_train.to_csv(os.path.join(PROJECT_ROOT, "outputs", "TF-IDF", "y_train.csv"), index=False)
y_val.to_csv(os.path.join(PROJECT_ROOT, "outputs", "TF-IDF", "y_val.csv"), index=False)
y_test.to_csv(os.path.join(PROJECT_ROOT, "outputs", "TF-IDF", "y_test.csv"), index=False)

print("Saved Vectorizer and TF-IDF matrices successfully!")